In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import joblib
import os
import json
warnings.filterwarnings("ignore")

# Sklearn — Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, r2_score,
)

# Sklearn — Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

In [2]:


df = pd.read_csv('C:\\Users\ASUS\\Downloads\\balanced_blood_pressure_dataset_v2.csv')
print(f"  Rows    : {df.shape[0]:,}")
print(f"  Columns : {df.shape[1]}")
print("=" * 55)
print(df.head(5).to_string())

# Column info
print("\n📊 COLUMN TYPES & NULL COUNTS")
print("-" * 40)
info = pd.DataFrame({
    "dtype":  df.dtypes,
    "nulls":  df.isnull().sum(),
    "unique": df.nunique(),
})
print(info)
print()
print("📊 NUMERICAL STATISTICS")
print(df.describe().round(2).to_string())

  Rows    : 2,800
  Columns : 12
   Patient_ID  Age Gender   BMI Smoking   Alcohol Physical_Activity  Cholesterol  Heart_Rate Stress_Level  Systolic_BP  Diastolic_BP
0           1   69      F  29.8     Yes       Low            Medium          195          77          Low          134            86
1           2   44      M  22.3      No  Moderate            Medium          196          64       Medium          127            77
2           3   59      M  30.2      No       Low              High          202          80         High          139            82
3           4   19      M  22.2      No       Low              High          189          74          Low          111            79
4           5   33      M  28.0     Yes       Low            Medium          201          73         High          123            68

📊 COLUMN TYPES & NULL COUNTS
----------------------------------------
                     dtype  nulls  unique
Patient_ID           int64      0    2800
Age           

In [3]:
plt.rcParams.update({
    "figure.facecolor": "#050810",
    "axes.facecolor":   "#0d1428",
    "axes.edgecolor":   "#1a2540",
    "axes.labelcolor":  "#e2e8f0",
    "text.color":       "#e2e8f0",
    "xtick.color":      "#64748b",
    "ytick.color":      "#64748b",
    "grid.color":       "#1a2540",
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
    "font.family":      "monospace",
})

COLORS = ["#00f5ff", "#ff2d78", "#a855f7", "#f59e0b", "#22c55e"]

In [4]:
import os
os.makedirs("static", exist_ok=True)
os.makedirs("models", exist_ok=True)

In [5]:
# 2. EDA PLOTS
# =============================================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Blood Pressure Dataset — EDA", fontsize=16,
             color="#00f5ff", fontweight="bold", y=1.02)

axes[0, 0].hist(df["Age"], bins=20, color="#00f5ff", alpha=0.8, edgecolor="#050810")
axes[0, 0].set_title("Age Distribution", color="#00f5ff")
axes[0, 0].set_xlabel("Age"); axes[0, 0].grid(True)

axes[0, 1].hist(df["BMI"], bins=20, color="#a855f7", alpha=0.8, edgecolor="#050810")
axes[0, 1].set_title("BMI Distribution", color="#a855f7")
axes[0, 1].set_xlabel("BMI"); axes[0, 1].grid(True)

axes[0, 2].hist(df["Systolic_BP"], bins=25, color="#ff2d78", alpha=0.8, edgecolor="#050810")
axes[0, 2].set_title("Systolic BP Distribution", color="#ff2d78")
axes[0, 2].set_xlabel("Systolic BP (mmHg)"); axes[0, 2].grid(True)

gcounts = df["Gender"].value_counts()
axes[1, 0].pie(gcounts, labels=gcounts.index, autopct="%1.1f%%",
               colors=["#00f5ff", "#ff2d78"], startangle=90,
               textprops={"color": "#e2e8f0"})
axes[1, 0].set_title("Gender Distribution", color="#e2e8f0")

axes[1, 1].hist(df["Cholesterol"], bins=20, color="#f59e0b", alpha=0.8, edgecolor="#050810")
axes[1, 1].set_title("Cholesterol Distribution", color="#f59e0b")
axes[1, 1].set_xlabel("Cholesterol (mg/dL)"); axes[1, 1].grid(True)

num_cols = ["Age", "BMI", "Cholesterol", "Heart_Rate", "Systolic_BP", "Diastolic_BP"]
corr = df[num_cols].corr()
sns.heatmap(corr, ax=axes[1, 2], cmap="coolwarm", annot=True, fmt=".2f",
            linewidths=0.5, linecolor="#050810",
            annot_kws={"size": 8}, cbar_kws={"shrink": 0.8})
axes[1, 2].set_title("Feature Correlation", color="#e2e8f0")

plt.tight_layout()
plt.savefig("static/eda_plots.png", dpi=120, bbox_inches="tight", facecolor="#050810")
plt.close()
print("\n✅ EDA plots saved → static/eda_plots.png")


✅ EDA plots saved → static/eda_plots.png


In [6]:
# 3. PREPROCESSING
# =============================================================================

# --- Target variable ---
def bp_category(row):
    s, d = row["Systolic_BP"], row["Diastolic_BP"]
    if s < 120 and d < 80:   return "Normal"
    elif s < 130 and d < 80: return "Elevated"
    elif s < 140 or d < 90:  return "High Stage 1"
    else:                    return "High Stage 2"

df["BP_Category"] = df.apply(bp_category, axis=1)
print("\n📊 BP Category Distribution:")
print(df["BP_Category"].value_counts())

# --- Label Encoding ---
label_encoders = {}
categorical_cols = ["Gender", "Smoking", "Alcohol", "Physical_Activity", "Stress_Level"]

for col in categorical_cols:
    le = LabelEncoder()
    df[col + "_enc"] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"  {col:20s} → {dict(zip(le.classes_, le.transform(le.classes_)))}")

le_target = LabelEncoder()
df["BP_Cat_enc"] = le_target.fit_transform(df["BP_Category"])
label_encoders["BP_Category"] = le_target
print(f"\n  BP_Category classes : {list(le_target.classes_)}")
print(f"  BP_Category encoded : {list(le_target.transform(le_target.classes_))}")

# --- Feature selection & split ---
FEATURES = [
    "Age", "Gender_enc", "BMI", "Smoking_enc",
    "Alcohol_enc", "Physical_Activity_enc",
    "Cholesterol", "Heart_Rate", "Stress_Level_enc",
]

X      = df[FEATURES]
y_cls  = df["BP_Cat_enc"]
y_sys  = df["Systolic_BP"]

X_train, X_test, y_cls_train, y_cls_test = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)
_, _, y_sys_train, y_sys_test = train_test_split(
    X, y_sys, test_size=0.2, random_state=42
)

scaler       = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train)
X_test_sc    = scaler.transform(X_test)

print(f"\n  Total samples  : {len(X):,}")
print(f"  Training set   : {len(X_train):,}  ({len(X_train)/len(X)*100:.0f}%)")
print(f"  Test set       : {len(X_test):,}   ({len(X_test)/len(X)*100:.0f}%)")
print(f"  Features used  : {len(FEATURES)}")


📊 BP Category Distribution:
BP_Category
High Stage 1    700
Elevated        700
Normal          700
High Stage 2    700
Name: count, dtype: int64
  Gender               → {'F': 0, 'M': 1}
  Smoking              → {'No': 0, 'Yes': 1}
  Alcohol              → {'High': 0, 'Low': 1, 'Moderate': 2}
  Physical_Activity    → {'High': 0, 'Low': 1, 'Medium': 2}
  Stress_Level         → {'High': 0, 'Low': 1, 'Medium': 2}

  BP_Category classes : ['Elevated', 'High Stage 1', 'High Stage 2', 'Normal']
  BP_Category encoded : [0, 1, 2, 3]

  Total samples  : 2,800
  Training set   : 2,240  (80%)
  Test set       : 560   (20%)
  Features used  : 9


In [7]:
# 4. MODEL TRAINING
# =============================================================================

# Model 1 — Decision Tree
print("\n🌳 Training Decision Tree Classifier ...")
dt_model = DecisionTreeClassifier(max_depth=8, min_samples_split=10,
                                   min_samples_leaf=4, random_state=42)
dt_model.fit(X_train, y_cls_train)
dt_pred = dt_model.predict(X_test)
dt_acc  = accuracy_score(y_cls_test, dt_pred)
print(f"   ✅ Accuracy : {dt_acc*100:.2f}%")
print(classification_report(y_cls_test, dt_pred, target_names=le_target.classes_))





# Model 2 — Naive Bayes
print("\n🎲 Training Gaussian Naive Bayes ...")
nb_model = GaussianNB()
nb_model.fit(X_train_sc, y_cls_train)
nb_pred = nb_model.predict(X_test_sc)
nb_acc  = accuracy_score(y_cls_test, nb_pred)
print(f"   ✅ Accuracy : {nb_acc*100:.2f}%")
print(classification_report(y_cls_test, nb_pred, target_names=le_target.classes_))

# Model 3 — Gradient Boosting  ⭐ BEST
print("🚀 Training Gradient Boosting Classifier ...")
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                       max_depth=5, random_state=42)
gb_model.fit(X_train, y_cls_train)
gb_pred = gb_model.predict(X_test)
gb_acc  = accuracy_score(y_cls_test, gb_pred)
print(f"   ✅ Accuracy : {gb_acc*100:.2f}%  ← BEST MODEL")
print(classification_report(y_cls_test, gb_pred, target_names=le_target.classes_))

# Model 4 — Random Forest
print("🌲 Training Random Forest Classifier ...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=None,
                                   min_samples_split=5, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_cls_train)
rf_pred = rf_model.predict(X_test)
rf_acc  = accuracy_score(y_cls_test, rf_pred)
print(f"   ✅ Accuracy : {rf_acc*100:.2f}%")
print(classification_report(y_cls_test, rf_pred, target_names=le_target.classes_))


🌳 Training Decision Tree Classifier ...
   ✅ Accuracy : 85.00%
              precision    recall  f1-score   support

    Elevated       0.76      0.86      0.81       140
High Stage 1       0.83      0.74      0.78       140
High Stage 2       0.97      0.81      0.89       140
      Normal       0.87      0.99      0.92       140

    accuracy                           0.85       560
   macro avg       0.86      0.85      0.85       560
weighted avg       0.86      0.85      0.85       560


🎲 Training Gaussian Naive Bayes ...
   ✅ Accuracy : 78.39%
              precision    recall  f1-score   support

    Elevated       0.72      0.74      0.73       140
High Stage 1       0.70      0.66      0.68       140
High Stage 2       0.84      0.83      0.83       140
      Normal       0.86      0.91      0.89       140

    accuracy                           0.78       560
   macro avg       0.78      0.78      0.78       560
weighted avg       0.78      0.78      0.78       560

🚀 Trai

In [8]:
# 5. EVALUATION PLOTS
# =============================================================================
results = {
    "Decision Tree":     {"accuracy": dt_acc*100, "metric": "Accuracy",  "type": "Classification"},
    "Naive Bayes":       {"accuracy": nb_acc*100, "metric": "Accuracy",  "type": "Classification"},
    "Gradient Boosting": {"accuracy": gb_acc*100, "metric": "Accuracy",  "type": "Classification"},
    "Random Forest":     {"accuracy": rf_acc*100, "metric": "Accuracy",  "type": "Classification"},
}

print("\n" + "=" * 62)
print(f"  {'ALGORITHM':<22} {'TYPE':<16} {'METRIC':<12} {'SCORE':>8}")
print("=" * 62)
sorted_results = sorted(results.items(), key=lambda x: -x[1]["accuracy"])
for i, (name, info) in enumerate(sorted_results):
    star = "  ⭐ BEST" if i == 0 else ""
    print(f"  {name:<22} {info['type']:<16} {info['metric']:<12} {info['accuracy']:>7.2f}%{star}")
print("=" * 62)

# Accuracy bar chart
names  = [r[0] for r in sorted_results]
scores = [r[1]["accuracy"] for r in sorted_results]
bar_colors = ["#f59e0b", "#22c55e", "#00f5ff", "#a855f7", "#ff2d78"]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(names, scores, color=bar_colors, edgecolor="#050810", height=0.55)
for bar, score, c in zip(bars, scores, bar_colors):
    ax.text(score + 0.3, bar.get_y() + bar.get_height()/2,
            f"{score:.2f}%", va="center", ha="left", color=c, fontsize=11, fontweight="bold")
ax.set_xlim(60, 95)
ax.set_xlabel("Score (%)", fontsize=11)
ax.set_title("Model Accuracy / R² Comparison", fontsize=14, color="#00f5ff", fontweight="bold", pad=16)
ax.axvline(x=80, color="#1a2540", linestyle="--", linewidth=1)
ax.grid(axis="x", alpha=0.3)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("static/accuracy_comparison.png", dpi=120, bbox_inches="tight", facecolor="#050810")
plt.close()
print("✅ Saved: static/accuracy_comparison.png")

# Confusion matrices
class_names = le_target.classes_
cms = {
    "Decision Tree":     (dt_pred, "#00f5ff"),
    "Naive Bayes":       (nb_pred, "#a855f7"),
    "Gradient Boosting": (gb_pred, "#f59e0b"),
    "Random Forest":     (rf_pred, "#22c55e"),
}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Confusion Matrices — All Classifiers", fontsize=15,
             color="#e2e8f0", fontweight="bold")
for ax, (name, (pred, color)) in zip(axes.flatten(), cms.items()):
    cm = confusion_matrix(y_cls_test, pred)
    sns.heatmap(cm, ax=ax, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names,
                linewidths=0.5, linecolor="#050810", annot_kws={"size": 10})
    acc = accuracy_score(y_cls_test, pred)
    ax.set_title(f"{name}  [{acc*100:.2f}%]", color=color, fontsize=11)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("static/confusion_matrices.png", dpi=120, bbox_inches="tight", facecolor="#050810")
plt.close()
print("✅ Saved: static/confusion_matrices.png")

# Feature importance
DISPLAY_NAMES = [
    "Age", "Gender", "BMI", "Smoking",
    "Alcohol", "Physical Activity",
    "Cholesterol", "Heart Rate", "Stress Level",
]
rf_fi = rf_model.feature_importances_
gb_fi = gb_model.feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (fi, title, color) in zip(axes, [
    (rf_fi, "Random Forest", "#22c55e"),
    (gb_fi, "Gradient Boosting", "#f59e0b"),
]):
    idx = np.argsort(fi)
    sorted_names = [DISPLAY_NAMES[i] for i in idx]
    sorted_fi    = fi[idx]
    cbar = plt.cm.cool(np.linspace(0.2, 0.8, len(sorted_fi)))
    ax.barh(sorted_names, sorted_fi, color=cbar, edgecolor="#050810")
    for val, y_pos in zip(sorted_fi, range(len(sorted_fi))):
        ax.text(val + 0.003, y_pos, f"{val:.3f}", va="center", color="#e2e8f0", fontsize=9)
    ax.set_title(f"Feature Importance — {title}", color=color, fontsize=12)
    ax.set_xlabel("Importance Score")
    ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("static/feature_importance.png", dpi=120, bbox_inches="tight", facecolor="#050810")
plt.close()
print("✅ Saved: static/feature_importance.png")


  ALGORITHM              TYPE             METRIC          SCORE
  Gradient Boosting      Classification   Accuracy       85.18%  ⭐ BEST
  Decision Tree          Classification   Accuracy       85.00%
  Random Forest          Classification   Accuracy       84.82%
  Naive Bayes            Classification   Accuracy       78.39%
✅ Saved: static/accuracy_comparison.png
✅ Saved: static/confusion_matrices.png
✅ Saved: static/feature_importance.png


In [9]:
# 6. SAVE MODELS
# =============================================================================
os.makedirs("models", exist_ok=True)

joblib.dump(dt_model,       "models/decision_tree.pkl")

joblib.dump(nb_model,       "models/naive_bayes.pkl")
joblib.dump(gb_model,       "models/gradient_boosting.pkl")
joblib.dump(rf_model,       "models/random_forest.pkl")
joblib.dump(scaler,         "models/scaler.pkl")
joblib.dump(label_encoders, "models/label_encoders.pkl")
joblib.dump(le_target,      "models/le_target.pkl")

accuracy_info = {
    "dt": {"name": "Decision Tree",     "accuracy": round(dt_acc*100, 2), "metric": "Accuracy"},
 
    "nb": {"name": "Naive Bayes",       "accuracy": round(nb_acc*100, 2), "metric": "Accuracy"},
    "gb": {"name": "Gradient Boosting", "accuracy": round(gb_acc*100, 2), "metric": "Accuracy"},
    "rf": {"name": "Random Forest",     "accuracy": round(rf_acc*100, 2), "metric": "Accuracy"},
}
with open("models/accuracy_info.json", "w") as f:
    json.dump(accuracy_info, f, indent=2)

print("\n✅ All models saved to ./models/")
for fname in sorted(os.listdir("models")):
    size = os.path.getsize(f"models/{fname}")
    print(f"   {fname:<35}  {size/1024:>7.1f} KB")

print("\n🎉 model.py complete! Now run:  python app.py")


✅ All models saved to ./models/
   accuracy_info.json                       0.4 KB
   decision_tree.pkl                        8.0 KB
   gradient_boosting.pkl                 2315.5 KB
   label_encoders.pkl                       2.0 KB
   le_target.pkl                            0.6 KB
   linear_regression.pkl                    0.7 KB
   naive_bayes.pkl                          1.4 KB
   random_forest.pkl                     4377.4 KB
   scaler.pkl                               1.2 KB

🎉 model.py complete! Now run:  python app.py
